# Denoised perturbation results: immune and cancer biology

The all-gene bidirectional screen returns 3,586 qualified gene-comparison hits, but the
top of the ranking is occupied by ribosomal, mitochondrial, heat-shock and ambient-RNA
transcripts. Those genes can be statistically extreme because they track global
transcriptional output, dissociation stress, or contamination from neighbouring cells —
not because they are T-cell-intrinsic regulators.

This notebook applies a transparent, auditable classification to every qualified hit and
re-ranks the results within curated immune and cancer programs.

## TL;DR

Of 3,586 qualified gene-comparison rows:

- **432 (12%)** match an explicit technical class (ribosomal, mitochondrial, heat-shock, immediate-early, ambient) and are set aside.
- **293 (8%)**, covering **110 genes**, map to a curated immune or cancer program. This is the prioritized set.
- **2,837 (79%)** are unannotated here — neither flagged as technical nor claimed as immune.

This is therefore **positive selection**, not just noise subtraction: the large `other` tier is not
asserted to be noise, it is simply outside the curated sets and left available for review.

Once ribosomal genes no longer anchor the score, **antigen presentation / MHC becomes the strongest
surviving program** — `HLA-C`, `B2M`, `CD74`, `HLA-B`, `HLA-E`, `PSMB9` occupy most of the top 20,
with detection counts in the thousands.

Filtering changes which genes are *prioritized*. It does not add evidence: donor-level
consistency and ambient-RNA sensitivity analysis are still required before any causal claim.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import Image, display

HERE = Path.cwd()
TABLES = HERE / 'tables'
FIGURES = HERE / 'figures' / 'denoised'

classified = pd.read_csv(TABLES / 'denoised_all_classified.csv')
immune = pd.read_csv(TABLES / 'immune_cancer_candidates.csv')
audit = pd.read_csv(TABLES / 'denoise_audit.csv')
recurrence = pd.read_csv(TABLES / 'immune_cancer_recurrence.csv')
programs = pd.read_csv(TABLES / 'immune_cancer_program_summary.csv')
print(f'Qualified rows: {len(classified):,}')
classified.tier.value_counts().rename('rows').to_frame()

## What is removed, and why

Each class below is a stated, reviewable reason for exclusion rather than an opaque filter.
The curated immune and ambiguous sets are matched **before** the noise patterns, so a real
immune gene can never be discarded by a regex — `HLA-*`, `B2M` and `CD74` survive by construction.

| Class | Reason it is set aside |
|---|---|
| Ribosomal, OXPHOS, structural housekeeping | Track global transcriptional output and cell size, not a specific program |
| Mitochondrial (`MT-`) | Dominated by cell stress and dying-cell fraction |
| Heat shock / proteostasis | Tissue handling and dissociation temperature artifacts |
| Immediate-early (`JUN`, `FOS`, `EGR1`, `DUSP1`) | Induced by the dissociation protocol itself |
| Epithelial, myeloid, platelet, hemoglobin | Ambient RNA from neighbouring cells in the tumour |
| Histone, non-coding | Cell-cycle and technical capture effects |

The `other` tier is the largest and is **not** a noise claim. It holds genes with no membership in
the curated sets used here. Real regulators certainly sit in it — that is the cost of positive
selection, and the reason the full classified table is written out rather than discarded.

In [ ]:
audit

In [ ]:
display(Image(filename=str(FIGURES / 'tier_composition_by_comparison.png')))

## Prioritized immune and cancer candidates

The balanced score is recomputed **within** the denoised set. This matters: the original
normalization was anchored to ribosomal genes that defined the top of each comparison, so
immune genes were compressed into the bottom of the scale.

In [ ]:
programs

In [ ]:
cols = ['comparison', 'Gene_name', 'class_label', 'direction', 'delete_shift', 'overexpress_shift', 'min_detection', 'denoised_score']
immune.sort_values('denoised_score', ascending=False)[cols].head(30)

In [ ]:
display(Image(filename=str(FIGURES / 'immune_cancer_heatmap.png')))

## Directionality

Sign convention follows the rest of this project: a **positive deletion shift** means removing
the gene moves cells *toward* the goal state, so the gene normally **restrains** that transition
and helps maintain the source identity. A negative deletion shift means the gene **promotes**
the transition. Concordance requires the overexpression arm to point the opposite way.

This is the same logic that made `ASCL1` / `NEUROD1` readable as SCLC master regulators.

In [ ]:
display(Image(filename=str(FIGURES / 'program_directionality.png')))

In [ ]:
immune.groupby(['class_label', 'direction']).size().unstack(fill_value=0).assign(
    total=lambda d: d.sum(axis=1)
).sort_values('total', ascending=False)

## Recurrence across disease transitions

A candidate seen in one transition can be comparison-specific noise. A candidate concordant
across several transitions is harder to explain that way. `n_restrains` and `n_promotes` show
whether the gene acts consistently or flips direction depending on the transition — a flip is
biologically meaningful for a state-discriminating gene, not a contradiction.

In [ ]:
recurrence.head(25)

In [ ]:
display(Image(filename=str(FIGURES / 'immune_cancer_recurrence.png')))

In [ ]:
display(Image(filename=str(FIGURES / 'immune_cancer_bidirectional.png')))

## Cross-check against the targeted 50-gene panel

The earlier curated panel was run as a separate, hypothesis-driven experiment. Genes that were
prioritized there and also surface here — found blind in an unbiased genome-wide screen —
are the strongest internal corroboration available without new data.

In [ ]:
PANEL = ['TIGIT', 'LAG3', 'GZMH', 'CCR7', 'NKG7', 'TCF7', 'IL7R', 'HAVCR2', 'CTLA4', 'SLAMF6', 'IFNG', 'PDCD1', 'ASCL1', 'NEUROD1']
panel_hits = immune[immune.Gene_name.isin(PANEL)]
summary = panel_hits.groupby('Gene_name').agg(
    n_comparisons=('comparison', 'nunique'),
    programs=('class_label', 'first'),
    median_detection=('min_detection', 'median'),
    max_score=('denoised_score', 'max'),
).sort_values('n_comparisons', ascending=False)
missing = sorted(set(PANEL) - set(panel_hits.Gene_name))
print('Panel genes not concordant anywhere in the all-gene screen:', missing or 'none')
summary

## Ambiguous tier

These genes are immune-meaningful and dissociation-induced at the same time. Assigning them to
either side would be a hidden judgement call, so they are kept visible and excluded from the
prioritized set by default.

In [ ]:
cols = ['comparison', 'Gene_name', 'direction', 'delete_shift', 'overexpress_shift', 'min_detection', 'balanced_score']
classified[classified.tier.eq('ambiguous')].sort_values('balanced_score', ascending=False)[cols]

## Gene lookup

Inspect any gene across all six transitions, including ones that were filtered out.

In [ ]:
def gene_lookup(gene):
    cols = ['comparison', 'Gene_name', 'tier', 'class_label', 'direction', 'delete_shift', 'overexpress_shift', 'min_detection']
    return classified[classified.Gene_name.eq(gene)].sort_values('comparison')[cols]

gene_lookup('HLA-C')

## Caveats

- **Filtering is a prioritization step, not evidence.** Every surviving candidate carries exactly the
  same statistical support it had before; the set is smaller and more interpretable, not more proven.
- **Gene-set membership is curated by symbol** in `scripts/build_denoised_notebook.py`. It is transparent
  and editable, but it is not GO/Reactome enrichment and it encodes assumptions about what counts as immune.
- **Excluded does not mean biologically inert.** Ribosomal and MHC-adjacent proteostasis genes have real
  roles in T-cell function; they are set aside because this assay cannot separate that from global state.
- **Ambient RNA is flagged by symbol, not corrected.** A CellBender run is still the right fix.
- **Donor-level consistency is still not applied to this all-gene result** — the single largest remaining gap.
  Until it is, no candidate here should be promoted to a biological claim in an abstract or poster.